# Лабораторна робота № 4. Локальний запуск і оптимізація моделей

**Виконав: Некрасов Артур Дмитрович**  
**Дедлайн: 14.11.2026**

Демо лекції збережено нижче. Додано відповіді на 20 питань, експерименти й звіт 1.8.
Локальні вимірювання в цьому файлі стосуються окремого середовища виконання Linux з CPU, а не особистого комп’ютера студента.
Оригінальні GPU-клітинки не виконано в цьому середовищі: їх потрібно запустити в Google Colab з T4. Числові висновки до цих клітинок обчислюються з результатів запуску. Сліпі оцінки вводяться після перегляду відповідей A/B.


# Лекція 5. Локальний деплоймент і оптимізація моделей

Демо до лекції. Усі моделі в цьому ноутбуці запускаються **локально** через [Ollama](https://ollama.com), їх використання відбувається через HTTP-запити.

## 0. Налаштування середовища

### 0.0. Імпортуємо бібліотеки

In [ ]:
import json
import subprocess
import sys
import time
import urllib.request

OLLAMA_URL = "http://localhost:11434"

### 0.1. Встановлюємо й запускаємо Ollama

Локально цю клітинку можна просто виконати: вона нічого не зробить.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !sudo apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

### 0.2. Перевіряємо, що Ollama працює

Ollama — це HTTP-сервер на порту 11434. Функція `ollama()` надсилає йому запит і повертає відповідь як словник Python; якщо сервер працює, побачимо його версію.

In [ ]:
def ollama(path, body=None, timeout=900):
    """POST-запит до Ollama (або GET, якщо body=None); повертає розібраний JSON."""
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(OLLAMA_URL + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.load(resp)


print("Ollama version:", ollama("/api/version")["version"])

## 1. Base, Instruct і Thinking

Порівнюємо три поведінки однієї й тієї самої моделі Qwen3 1.7B:
- **Base-модель** пройшла лише попереднє тренування: вона вміє передбачати наступний токен в інтернет-текстах, але не вміє відповідати на питання як асистент.
- **Instruct-модель** додатково донавчили на прикладах «інструкція → відповідь» у форматі діалогу.
- **Thinking** — режим, у якому Instruct-модель перед відповіддю генерує ланцюжок міркувань.

### 1.0. Завантажуємо моделі

Беремо одну й ту саму модель Qwen3 1.7B у двох версіях:
- **Base** — [Qwen3-1.7B-Base](https://huggingface.co/Qwen/Qwen3-1.7B-Base) у форматі GGUF (Q4_K_M) з Hugging Face;
- **Instruct** — офіційна [qwen3:1.7b](https://ollama.com/library/qwen3) з каталогу Ollama; вона ж уміє працювати в режимі міркування.

Перше завантаження займає кілька хвилин.

In [ ]:
BASE_MODEL = "hf.co/mradermacher/Qwen3-1.7B-Base-GGUF:Q4_K_M"
INSTRUCT_MODEL = "qwen3:1.7b"

for m in (BASE_MODEL, INSTRUCT_MODEL):
    print("pull", m, "→", ollama("/api/pull", {"model": m, "stream": False})["status"])

### 1.1. Допоміжні функції

Нам вистачить двох ендпойнтів Ollama:
- `/api/generate` з `"raw": true` — передає моделі текст як є, **без chat template**;
- `/api/chat` — приймає список повідомлень і сам загортає їх у chat template моделі.

У Qwen3 режим міркування вмикається параметром `think`. З `think=True` модель спершу генерує ланцюжок міркувань (Ollama повертає його окремим полем `message.thinking`), а вже потім — відповідь. Функція `compare_thinking()` запускає запит в обох режимах і показує відповідь, кількість згенерованих токенів і час.

In [ ]:
THINK_RUNS = []


def generate_raw(model, prompt, max_tokens=120):
    r = ollama("/api/generate", {
        "model": model, "prompt": prompt, "raw": True, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })
    return r["response"]


def chat(model, prompt, think=False, max_tokens=3000):
    return ollama("/api/chat", {
        "model": model, "messages": [{"role": "user", "content": prompt}],
        "think": think, "stream": False,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens},
    })


def measured_chat(prompt, think=False, max_tokens=3000):
    body = {"model": INSTRUCT_MODEL, "messages": [{"role": "user", "content": prompt}],
        "think": think, "stream": True,
        "options": {"temperature": 0, "seed": 1, "num_predict": max_tokens}}
    req = urllib.request.Request(OLLAMA_URL + "/api/chat", data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"})
    started = time.perf_counter()
    first_thinking = first_content = None
    final_text, thinking_parts = [], []
    last = {}
    with urllib.request.urlopen(req, timeout=900) as response:
        for line in response:
            item = json.loads(line)
            msg = item.get("message", {})
            if msg.get("thinking"):
                if first_thinking is None:
                    first_thinking = time.perf_counter()
                thinking_parts.append(msg["thinking"])
            if msg.get("content"):
                if first_content is None:
                    first_content = time.perf_counter()
                final_text.append(msg["content"])
            last = item
    ended = time.perf_counter()
    thinking_text = "".join(thinking_parts)
    record = {"prompt": prompt, "think": think, "answer": "".join(final_text),
        "eval_count": last.get("eval_count"),
        "eval_s": last.get("eval_duration", 0) / 1e9,
        "wall_s": ended - started,
        "ttft_s": None if first_content is None else first_content - started,
        "thinking_phase_s": None if first_thinking is None or first_content is None else first_content - first_thinking,
        "answer_phase_s": None if first_content is None else ended - first_content,
        "thinking_characters": len(thinking_text),
        "thinking_latin_characters": sum(ch.isascii() and ch.isalpha() for ch in thinking_text),
        "_thinking_text": thinking_text}
    THINK_RUNS.append(record)
    return record


def compare_thinking(prompt):
    for think in (False, True):
        r = measured_chat(prompt, think)
        print(f"think={think}; токенів загалом: {r['eval_count']}; генерація: {r['eval_s']:.2f} с; TTFT: {r['ttft_s']}")
        print(r["answer"])
        print()


### 1.2. Base: модель просто продовжує текст

Поставимо Base- та Instruct-моделі однакове питання.

In [ ]:
QUESTION = "Яка столиця Франції?"

In [ ]:
print(generate_raw(BASE_MODEL, QUESTION))

Для Base-моделі «Яка столиця Франції?» схоже на початок списку питань вікторини — тож вона продовжує цей список, а не відповідає.

### 1.3. Instruct: модель відповідає на питання

In [ ]:
print(chat(INSTRUCT_MODEL, QUESTION)["message"]["content"])

### 1.4. Instruct проти Thinking: багатокрокова задача

In [ ]:
compare_thinking(
    "У кошику 3 яблука. Марія поклала туди ще 2 пакети по 4 яблука, "
    "а потім зʼїла 2 яблука. Скільки яблук у кошику?"
)

Обидва режими відповідають правильно, але з міркуванням модель генерує в кілька разів більше токенів — а користувач бачить відповідь лише після них. Зверніть увагу: міркує модель англійською, хоча питання українською.

### 1.5. Міркування не гарантують правильності

In [ ]:
compare_thinking("Скільки разів літера «р» трапляється в слові «трансформер»?")

Правильна відповідь — 3 (т**р**ансфо**р**ме**р**), але мала модель помиляється в обох режимах. Чому так — подумайте самі (це питання до лабораторної роботи).

Спробуйте власні запити: де режим міркування справді допомагає, а де лише витрачає токени?

## 2. Квантизація з Quanto

Завантажимо ту саму модель у чотирьох варіантах — від оригінальних 16 біт до 2 біт — і подивимося, що квантизація змінює в памʼяті, швидкості та якості. Якість перевіримо трьома способами: перплексією, перекладом з автоматичними метриками й очима — на вільній генерації тексту.

**Quanto.** [Optimum Quanto](https://github.com/huggingface/optimum-quanto) — бібліотека квантизації від Hugging Face, один із бекендів [Optimum](https://huggingface.co/docs/optimum). У Transformers вона підключається через [`QuantoConfig`](https://huggingface.co/docs/transformers/quantization/quanto) і квантизує модель прямо під час завантаження, тому окремий файл з квантизованими вагами не потрібен. Квантизуються ваги лінійних шарів (`nn.Linear`) — у них більшість параметрів моделі, а ембединги, нормалізації та вихідний шар лишаються в 16 бітах.

| Варіант | Біт на вагу | Що це |
|---|---|---|
| fp16 | 16 | Оригінальні ваги, точка відліку |
| int8 | 8 | 8-бітні цілі числа з масштабом |
| int4 | 4 | 4-бітні цілі числа, ваги діляться на блоки з власними масштабами |
| int2 | 2 | 2-бітні цілі числа — лише 4 можливі значення на вагу; тут модель має зламатися |

**Обчислення.** Множити числа в цих форматах напряму T4 не вміє, тому перед множенням ваги розпаковуються в 16 біт. Тому памʼяті потрібно менше, а от генерація часто **не прискорюється**, а то й сповільнюється.

**Де запускати.** Секцію запускаємо **в Google Colab на GPU T4** (Runtime → Change runtime type → T4 GPU). Беремо формат **fp16**, а не bf16: T4 не підтримує bf16 апаратно, а для запуску готової моделі різниця між ними практично непомітна.

### 2.0. Встановлюємо бібліотеки й перевіряємо GPU

In [ ]:
for model_name in (BASE_MODEL, INSTRUCT_MODEL):
    ollama("/api/generate", {"model": model_name, "keep_alive": 0})
print("Моделі Ollama вивантажено перед вимірюванням VRAM у Transformers.")

In [ ]:
%pip uninstall -y -q diffusers  # optimum-quanto імпортує diffusers, якщо він є; передвстановлена в Colab версія несумісна з його torchao
%pip install -q "transformers==5.20.1" "accelerate==1.15.0" "optimum-quanto==0.2.7" "evaluate==0.4.6" "sacrebleu==2.6.0"

In [ ]:
import gc
import time
from types import SimpleNamespace

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from optimum.quanto.library.extensions import get_extension
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

assert torch.cuda.is_available(), "Увімкніть GPU: Runtime → Change runtime type → T4 GPU"
DEVICE = "cuda"
DTYPE = torch.float16
print("GPU:", torch.cuda.get_device_name())


def unpack_weights(packed, bits):
    """Розпаковує 4- чи 2-бітні значення, по кілька упакованих в одному байті."""
    values_per_byte = 8 // bits
    mask = 2**bits - 1
    return torch.cat([(packed >> (bits * i)) & mask for i in range(values_per_byte)]).to(torch.uint8)


# Quanto на GPU компілює власні CUDA-ядра, розраховані на A100 і новіші; на T4 ця компіляція падає.
# На T4 з усіх цих ядер потрібна лише розпаковка int4/int2-ваг, тож підставляємо її версію на чистому PyTorch.
if torch.cuda.get_device_capability()[0] < 8:
    get_extension("quanto_cuda")._lib = SimpleNamespace(unpack=unpack_weights)

### 2.1. Модель і варіанти квантизації

Беремо [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) — сильнішу модель того ж сімейства, що й у секції 1: 4B параметрів, лише Instruct-режим без міркувань. У fp16 вона займає ~8 GB і вміщається в T4 з його 16 GB.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

VARIANTS = {
    "fp16": None,
    "int8": QuantoConfig(weights="int8"),
    "int4": QuantoConfig(weights="int4"),
    "int2": QuantoConfig(weights="int2"),
}

### 2.2. Що міряємо

**Перплексія** — стандартна автоматична метрика якості квантизації (нею міряє, зокрема, llama.cpp). Модель читає фіксований текст, і для кожного токена ми дивимося, яку ймовірність вона йому давала. Перплексія — це, грубо кажучи, з якої кількості варіантів модель у середньому «вагалася»: чим вона менша, тим краще модель знає мову й тему. Еталонних відповідей тут не потрібно, а число змінюється плавно, тому видно навіть невеликі втрати. Порівнювати перплексію можна лише в межах однієї мови: англійська й українська токенізуються по-різному.

**Переклад EN → UK** — реалістичне завдання з генерацією тексту, для якого є еталонні переклади. Порівнюємо переклад моделі з еталоном двома стандартними метриками з бібліотеки [Evaluate](https://huggingface.co/docs/evaluate):
- [**chrF**](https://github.com/huggingface/evaluate/blob/main/metrics/chrf/README.md) — наскільки збігаються послідовності символів (0–100);
- [**BLEU**](https://github.com/huggingface/evaluate/tree/main/metrics/sacrebleu) — наскільки збігаються послідовності слів (0–100). Сувора метрика: синонім чи інший порядок слів уже знижують бал.

**Вільна генерація** — лист, пояснення, план, код. Тут еталону немає, тож оцінюємо очима в 2.5.

Крім якості, для кожного варіанта записуємо **памʼять моделі** на GPU, **пік VRAM** і **швидкість генерації** (tokens/s).

In [ ]:
PPL_TEXT = {
    "English": (
        "Photosynthesis is the process by which green plants, algae and some bacteria turn light energy into "
        "chemical energy. It takes place mainly in the leaves, inside small structures called chloroplasts, which "
        "contain the green pigment chlorophyll. Chlorophyll absorbs sunlight, and the plant uses this energy to "
        "combine carbon dioxide from the air with water from the soil. The result is glucose, a sugar that the "
        "plant uses for growth and as a store of energy, and oxygen, which is released into the atmosphere. "
        "Almost all life on Earth depends on photosynthesis: it produces the oxygen we breathe and forms the "
        "basis of most food chains."
    ),
    "Українська": (
        "Фотосинтез — це процес, під час якого зелені рослини, водорості та деякі бактерії перетворюють енергію "
        "світла на хімічну енергію. Він відбувається переважно в листках, усередині маленьких структур, які "
        "називають хлоропластами і які містять зелений пігмент хлорофіл. Хлорофіл поглинає сонячне світло, і "
        "рослина використовує цю енергію, щоб поєднати вуглекислий газ із повітря з водою із ґрунту. У результаті "
        "утворюються глюкоза — цукор, який рослина використовує для росту і як запас енергії, — та кисень, що "
        "виділяється в атмосферу. Майже все життя на Землі залежить від фотосинтезу: він дає кисень, яким ми "
        "дихаємо, і лежить в основі більшості харчових ланцюгів."
    ),
}

TRANSLATION = [
    # (речення англійською, еталонний переклад)
    ("The meeting has been moved to Thursday at three o'clock.", "Зустріч перенесли на четвер на третю годину."),
    ("Please send me the report by the end of the day.", "Будь ласка, надішліть мені звіт до кінця дня."),
    ("The library is closed on Sundays and public holidays.", "Бібліотека не працює в неділю та у святкові дні."),
    ("Our team fixed the bug, but the tests are still failing.", "Наша команда виправила помилку, але тести все ще не проходять."),
    ("It is going to rain tomorrow, so take an umbrella.", "Завтра буде дощ, тож візьміть парасольку."),
    ("The new model is smaller, but it answers almost as well as the old one.", "Нова модель менша, але відповідає майже так само добре, як стара."),
    ("I would like to book a table for two for seven o'clock tonight.", "Я хотів би забронювати столик на двох на сьому вечора."),
    ("The price includes breakfast and free Wi-Fi.", "У вартість входять сніданок і безкоштовний Wi-Fi."),
    ("Students must submit their lab work before the deadline.", "Студенти мають здати лабораторну роботу до кінцевого терміну."),
    ("If the server does not respond, restart it and check the logs.", "Якщо сервер не відповідає, перезапустіть його й перевірте логи."),
]
TRANSLATE_PROMPT = "Translate the following sentence into Ukrainian. Output only the translation.\n\n{}"

GEN_PROMPTS = [
    "Напиши короткий лист викладачеві з проханням перенести здачу лабораторної роботи на тиждень через хворобу.",
    "Поясни студентові першого курсу в 3–4 реченнях, що таке квантизація нейромережі.",
    "Explain in three sentences why the sky is blue.",
    "Склади план підготовки до іспиту з програмування на 5 днів, по одному пункту на день.",
    "Write a Python function that checks whether a string is a palindrome.",
]

### 2.3. Проганяємо всі варіанти

Дві допоміжні функції — перплексія тексту й генерація відповіді — і цикл по варіантах: завантажуємо модель, міряємо все потрібне, звільняємо памʼять. Займає кілька хвилин на варіант.

In [ ]:
from pathlib import Path
import random


def perplexity(model, text):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.inference_mode():
        loss = model(ids, labels=ids).loss
    return float(torch.exp(loss.float()))


def generate(model, prompt, max_new_tokens):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), len(new_tokens), time.perf_counter() - start


random_words = {}
for language, text in PPL_TEXT.items():
    words = text.split()
    rng_words = random.Random(1)
    random_words[language] = " ".join(rng_words.choices(words, k=len(words)))

records = {}
for name, config in VARIANTS.items():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE,
        device_map=DEVICE, quantization_config=config)
    model.eval()
    r = {"памʼять моделі, GB": torch.cuda.memory_allocated() / 1e9}
    if name == "fp16":
        PARAM_COUNT_4B = sum(p.numel() for p in model.parameters())
        r["кількість параметрів"] = PARAM_COUNT_4B
        r["випадкові слова, перплексія"] = {language: perplexity(model, text)
            for language, text in random_words.items()}
    for language, text in PPL_TEXT.items():
        r[f"перплексія, {language}"] = perplexity(model, text)
    r["переклади"] = [generate(model, TRANSLATE_PROMPT.format(src), 80)[0] for src, _ in TRANSLATION]
    generations = [generate(model, prompt, 200) for prompt in GEN_PROMPTS]
    r["генерації"] = [text for text, _, _ in generations]
    r["tokens/s"] = sum(n for _, n, _ in generations) / sum(s for _, _, s in generations)
    r["пік VRAM, GB"] = torch.cuda.max_memory_allocated() / 1e9
    records[name] = r
    Path("quantization_4b.json").write_text(json.dumps(records, ensure_ascii=False, indent=2))
    print(f"{name}: готово")
    del model


### 2.4. Памʼять, швидкість і автоматичні метрики

In [ ]:
chrf = evaluate.load("chrf")
bleu = evaluate.load("sacrebleu")
references = [[ref] for _, ref in TRANSLATION]

summary = pd.DataFrame({
    name: {
        "памʼять моделі, GB": r["памʼять моделі, GB"],
        "пік VRAM, GB": r["пік VRAM, GB"],
        "tokens/s": r["tokens/s"],
        **{f"перплексія, {lang}": r[f"перплексія, {lang}"] for lang in PPL_TEXT},
        "chrF перекладу": chrf.compute(predictions=r["переклади"], references=references)["score"],
        "BLEU перекладу": bleu.compute(predictions=r["переклади"], references=references)["score"],
    }
    for name, r in records.items()
}).T.round(2)
summary

Перплексія для кожної мови окремо. Вісь — логарифмічна: інакше на одному графіку не помістилися б і кілька одиниць, і сотні.

In [ ]:
LANG_COLORS = {"English": "#2a78d6", "Українська": "#eb6834"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

variants = list(records)
x = np.arange(len(variants))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 4.2))
for i, lang in enumerate(PPL_TEXT):
    values = summary[f"перплексія, {lang}"].values
    bars = ax.bar(x + (i - 0.5) * width, values, width - 0.04, color=LANG_COLORS[lang], label=lang)
    ax.bar_label(bars, labels=[f"{v:.1f}" for v in values], fontsize=8, color=MUTED, padding=2)
ax.set_yscale("log")
ax.set_xticks(x, variants)
ax.set_ylabel("перплексія (менше — краще)", color=INK)
ax.set_title("Перплексія на фіксованому тексті", loc="left", color=INK)
ax.legend(frameon=False, ncols=2, loc="upper left")
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

Переклади всіх варіантів поруч з еталоном:

In [ ]:
translations = pd.DataFrame({"еталон": [ref for _, ref in TRANSLATION]}, index=[src for src, _ in TRANSLATION])
for name, r in records.items():
    translations[name] = r["переклади"]
translations

### 2.5. Сліпе порівняння: fp16 проти int4

Автоматичні метрики показали, наскільки int4 відрізняється від fp16 у числах. Тепер перевірмо очима — так, як це роблять оцінювачі-люди: для кожного запиту показуємо дві відповіді, fp16 та int4, **у випадковому порядку й без підписів**. Ваше завдання — обрати кращу.

In [ ]:
rng = np.random.default_rng(0)
BLIND_ORDER = [rng.permutation(["fp16", "int4"]) for _ in GEN_PROMPTS]

for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for label, name in zip("AB", BLIND_ORDER[i]):
        display(Markdown(f"**Відповідь {label}:**\n\n{records[name]['генерації'][i]}"))

Для кожного запиту запишіть **"A"** або **"B"** — яка відповідь краща, або **"="**, якщо однакові за якістю. Потім запустіть наступну клітинку, щоб дізнатися, де була яка модель.

In [ ]:
VOTES = {}
for i in range(1, 6):
    vote = input(f"Запит {i}: A / B / =: ").strip().upper()
    while vote not in {"A", "B", "="}:
        vote = input("Введіть A, B або =: ").strip().upper()
    VOTES[i] = vote
Path("blind_votes.json").write_text(json.dumps(VOTES, ensure_ascii=False, indent=2))

In [ ]:
if all(v in ("A", "B", "=") for v in VOTES.values()):
    rows = []
    for i, vote in VOTES.items():
        order = dict(zip("AB", BLIND_ORDER[i - 1]))
        rows.append({"запит": GEN_PROMPTS[i - 1], "A": order["A"], "B": order["B"],
                     "ваш вибір": "однаково" if vote == "=" else order[vote]})
    display(pd.DataFrame(rows))
    choices = [r["ваш вибір"] for r in rows]
    print(f"fp16 краща: {choices.count('fp16')}, int4 краща: {choices.count('int4')}, однаково: {choices.count('однаково')}")
else:
    print('Заповніть VOTES значеннями "A", "B" або "=" для всіх 5 запитів.')

### 2.6. Відповіді всіх варіантів

Тепер, коли сліпе порівняння зроблено, — відповіді всіх варіантів з підписами, включно з int8 та int2. Еталону тут немає, тож читаємо самі.

In [ ]:
for i, prompt in enumerate(GEN_PROMPTS):
    display(Markdown(f"---\n#### Запит {i + 1}: {prompt}"))
    for name, r in records.items():
        display(Markdown(f"**{name}:**\n\n{r['генерації'][i]}"))

## 0.1. Відповіді на теоретичні питання

### 1. Літера «р» у слові «трансформер»

У слові три літери «р»: на позиціях 2, 8 і 11 при нумерації від одиниці. Модель отримує ідентифікатори токенів, а не послідовність окремих літер. Межі підслівних токенів не є межами символів; один токен може містити кілька літер. Підрахунок вимагає надійно відновити написання й виконати символьний алгоритм, якого генерація правдоподібного тексту не гарантує. Додаткові міркування також можуть повторити хибне розбиття. Токенізація пояснює ускладнення, але не робить правильний підрахунок неможливим. Надійний результат дає символьний підрахунок у Python.


In [1]:
from transformers import AutoTokenizer

letter_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
word = "трансформер"
positions = [i + 1 for i, letter in enumerate(word) if letter == "р"]
encoded = letter_tokenizer(word, add_special_tokens=False, return_offsets_mapping=True)
print("Символи:", list(word))
print("Позиції:", positions, "кількість:", word.count("р"))
for token_id, (start, end) in zip(encoded["input_ids"], encoded["offset_mapping"]):
    print(token_id, repr(letter_tokenizer.convert_ids_to_tokens(token_id)), repr(word[start:end]), (start, end))
assert word.count("р") == 3


Символи: ['т', 'р', 'а', 'н', 'с', 'ф', 'о', 'р', 'м', 'е', 'р']
Позиції: [2, 8, 11] кількість: 3
1792 'ÑĤ' 'т' (0, 1)
34582 'ÑĢÐ°Ð½' 'ран' (1, 4)
2247 'Ñģ' 'с' (4, 5)
56619 'ÑĦÐ¾ÑĢÐ¼' 'форм' (5, 9)
7599 'ÐµÑĢ' 'ер' (9, 11)


### 2. Перплексія й переклад від fp16 до int2

Менша перплексія означає краще передбачення токенів; більші BLEU і chrF означають більшу схожість перекладу з еталоном. int8 та int4 часто зберігають значну частину якості, а int2 може спричинити різке погіршення. Монотонність не гарантована на короткому наборі: округлення іноді випадково покращує окрему відповідь. Нижче з фактичної таблиці 2.4 обчислюються переходи з найбільшим зростанням перплексії та падінням метрик; очікування не підставляються замість вимірювань.


In [ ]:
ordered_variants = ["fp16", "int8", "int4", "int2"]
numeric_summary = summary.loc[ordered_variants].copy()
display(numeric_summary)
for metric in ["перплексія, English", "перплексія, Українська", "chrF перекладу", "BLEU перекладу"]:
    values = numeric_summary[metric]
    differences = values.diff().iloc[1:]
    worst = differences.idxmax() if metric.startswith("перплексія") else differences.idxmin()
    previous = ordered_variants[ordered_variants.index(worst) - 1]
    print(f"{metric}: найбільший шкідливий зсув {previous} → {worst}: {differences[worst]:+.2f}")


### 3. Англійська й українська

Порівнювати слід відносне зростання перплексії всередині кожної мови щодо її fp16, а не самі значення між мовами. Різні тексти, частоти та кількість токенів створюють різні базові рівні. Менш представлені під час навчання мови можуть сильніше втрачати якість при квантизації, як зазначено на слайді про вразливі задачі. Якщо зростання для української більше, експеримент узгоджується з цією тезою; один короткий текст на мову не доводить її для всіх задач.


In [ ]:
relative_ppl = pd.DataFrame({language: {variant: records[variant][f"перплексія, {language}"] /
    records["fp16"][f"перплексія, {language}"] for variant in ordered_variants} for language in PPL_TEXT})
display(relative_ppl.rename_axis("варіант").round(3))
for variant in ["int8", "int4", "int2"]:
    en, uk = relative_ppl.loc[variant, "English"], relative_ppl.loc[variant, "Українська"]
    print(variant, "відносне погіршення більше для", "української" if uk > en else "англійської" if en > uk else "обох однакове")


### 4. Сліпий вибір і автоматичні метрики

У секції 2.5 вибір вводиться без назв варіантів. Після розкриття назв потрібно порівняти частоту вибору fp16/int4 з різницею chrF, BLEU й перплексії. Метрики перекладу не оцінюють лист, план або правильність Python-функції; помітна стилістична чи фактична відмінність у цих задачах може не супроводжуватися зміною метрик перекладу. До введення п’яти власних оцінок особистий висновок не визначено.


In [ ]:
blind_choices = ["однаково" if VOTES[i] == "=" else dict(zip("AB", BLIND_ORDER[i-1]))[VOTES[i]]
    for i in range(1, 6)]
display(pd.DataFrame({"вибір": ["fp16", "int4", "однаково"],
    "кількість": [blind_choices.count(x) for x in ["fp16", "int4", "однаково"]]}))
display(summary.loc[["fp16", "int4"], ["chrF перекладу", "BLEU перекладу", "перплексія, English", "перплексія, Українська"]])
BLIND_OBSERVATION = input("Опишіть конкретну помічену відмінність і її зв’язок із метриками: ").strip()
print(BLIND_OBSERVATION)


### 5. Правильний переклад з іншими словами

BLEU спирається на збіг словесних n-грам і може занижувати оцінку коректного синонімічного перекладу, особливо з одним еталоном. chrF порівнює символьні n-грами, тому зазвичай м’якше ставиться до змін словоформи, але також не є перевіркою змісту й штрафує синоніми. У таблиці нижче є оцінки кожної фактичної відповіді. Потрібно вибрати змістовно правильний варіант, який відрізняється словами; сам факт відмінності не означає правильність.


In [ ]:
translation_scores = []
for variant, result in records.items():
    for i, ((source, reference), prediction) in enumerate(zip(TRANSLATION, result["переклади"]), 1):
        translation_scores.append({"речення": i, "варіант": variant, "еталон": reference,
            "переклад": prediction,
            "BLEU": bleu.compute(predictions=[prediction], references=[[reference]])["score"],
            "chrF": chrf.compute(predictions=[prediction], references=[[reference]])["score"]})
display(pd.DataFrame(translation_scores).round(2))
PARAPHRASE_VARIANT = input("Варіант правильного перекладу з іншими словами (fp16/int8/int4/int2): ").strip()
PARAPHRASE_INDEX = int(input("Номер речення (1–10): "))
chosen_translation = next(row for row in translation_scores if row["варіант"] == PARAPHRASE_VARIANT and row["речення"] == PARAPHRASE_INDEX)
display(pd.DataFrame([chosen_translation]).round(2))


### 6. Пам’ять і швидкість

Формула $N b / 8$ описує лише ідеальне зберігання квантизованих ваг. Масштаби, нульові точки, вирівнювання, неквантизовані embedding/нормалізації, буфери й тимчасові тензори не зменшуються в тих самих пропорціях. Тому пам’ять не обов’язково скорочується рівно у 2, 4 і 8 разів. Генерація прискорюється лише за наявності ефективних ядер і відповідного вузького місця: розпаковка int4/int2 на T4 у цьому демо може перекрити виграш від меншого читання ваг. Фактичні коефіцієнти обчислено нижче.


In [ ]:
display(pd.DataFrame({
    "пам’ять fp16 / варіант": summary.loc["fp16", "памʼять моделі, GB"] / summary["памʼять моделі, GB"],
    "швидкість / fp16": summary["tokens/s"] / summary.loc["fp16", "tokens/s"],
    "tokens/s": summary["tokens/s"]}).round(3))


### 7. Відповідь Base-моделі

Base-модель навчена продовжувати текст, а не виконувати роль асистента. Окреме питання схоже на початок переліку питань; продовження переліку є природним завершенням такого шаблону. Зразки «Питання — Відповідь» задають потрібний формат продовження. Це few-shot промпт, який не перетворює Base-модель на Instruct і не гарантує точність на всіх питаннях.


In [ ]:
gc.collect()
torch.cuda.empty_cache()
base_prompt = """Питання: Яка столиця України?
Відповідь: Київ.

Питання: Яка столиця Італії?
Відповідь: Рим.

Питання: Яка столиця Франції?
Відповідь:"""
base_answer = generate_raw(BASE_MODEL, base_prompt, max_tokens=40)
print(base_prompt)
print("Результат:", base_answer)
ollama("/api/generate", {"model": BASE_MODEL, "keep_alive": 0})


### 8. Вартість міркувань

У моделі один бюджет нових токенів витрачається і на міркування, і на фінальну відповідь. Для задачі про яблука нижче наведено серверні токени, час генерації та час до першого видимого токена; окреме розбиття токенів пораховано тим самим токенізатором і може відрізнятися на службові роздільники. Час фази міркування виміряний за потоком приблизно, бо фрагменти можуть об’єднуватися. Англійська мова внутрішніх міркувань пов’язана з навчальними даними й налаштуванням моделі: мова питання не гарантує мову прихованого процесу. Це не означає, що відповідь також має бути англійською.


In [ ]:
apple_runs = [r for r in THINK_RUNS if "У кошику 3 яблука" in r["prompt"]]
thinking_table = []
for r in apple_runs:
    thinking_table.append({"think": r["think"], "серверні токени разом": r["eval_count"],
        "токени міркування, текст": len(letter_tokenizer.encode(r["_thinking_text"], add_special_tokens=False)),
        "токени відповіді, текст": len(letter_tokenizer.encode(r["answer"], add_special_tokens=False)),
        "генерація, с": r["eval_s"], "TTFT, с": r["ttft_s"],
        "фаза міркування, с": r["thinking_phase_s"], "фаза відповіді, с": r["answer_phase_s"],
        "частка латинських літер у міркуванні": r["thinking_latin_characters"] / max(1, r["thinking_characters"])})
display(pd.DataFrame(thinking_table).round(3))
if len(apple_runs) == 2:
    plain, thinking = apple_runs
    print("Додаткові серверні токени:", thinking["eval_count"] - plain["eval_count"])
    print("Відношення часу генерації:", round(thinking["eval_s"] / max(plain["eval_s"], 1e-9), 3))


### 9. Задача, де міркування допомагає

Для пошуку використовується набір арифметичних задач із відповіддю, перевіреною Python. У результат зберігаються промпт, обидві фінальні відповіді та еталон. Приклад добирається за фактичною правильністю; за відсутності такого прикладу код повідомляє про це й не підставляє вигадані відповіді.


In [ ]:
import re

thinking_example = None
reasoning_trials = []
for a, b in [(173, 287), (137, 249), (1237, 8943), (987, 678), (2027, 391), (462, 759)]:
    prompt = f"Обчисли {a} × {b}. У фінальній відповіді напиши лише ціле число без пробілів."
    answers = [chat(INSTRUCT_MODEL, prompt, think=mode, max_tokens=3000)["message"]["content"].strip()
        for mode in (False, True)]
    expected = str(a * b)
    trial = {"prompt": prompt, "без міркувань": answers[0], "з міркуваннями": answers[1], "еталон": expected}
    reasoning_trials.append(trial)
    finals = [re.sub(r"[^0-9-]", "", text) for text in answers]
    if finals[0] != expected and finals[1] == expected:
        thinking_example = trial
        break
if thinking_example is None:
    print("У перевіреному наборі потрібного розходження не знайдено. Збільште набір задач.")
display(pd.DataFrame([thinking_example] if thinking_example else reasoning_trials))
Path("reasoning_trials.json").write_text(json.dumps(reasoning_trials, ensure_ascii=False, indent=2))
ollama("/api/generate", {"model": INSTRUCT_MODEL, "keep_alive": 0})


### 10. `temperature=0` і `seed=1`

Нульова температура задає жадібний вибір найімовірнішого наступного токена. Однаковий seed фіксує генератор випадкових чисел там, де рантайм використовує випадковість. Так порівняння краще відокремлює вплив моделі, квантизації й thinking від випадкового семплінгу. При жадібному декодуванні seed зазвичай не змінює результат. Абсолютна побітова відтворюваність між різними GPU, ядрами й версіями ПЗ не гарантується.

### 11. fp16 на T4

NVIDIA T4 має compute capability **7.5**; `torch.cuda.get_device_capability()` повертає `(7, 5)`. T4 підтримує ефективні fp16 операції Tensor Cores, але не має нативного bf16 Tensor Core виконання, яке з’явилося в Ampere. Тому для цього GPU обрано fp16. Наявність самого типу `torch.bfloat16` або програмної емуляції не означає апаратної підтримки bf16. Реальну характеристику поточної GPU показує клітинка нижче.


In [ ]:
print("GPU:", torch.cuda.get_device_name())
print("Compute capability:", torch.cuda.get_device_capability())
print("Нативна bf16-підтримка за поколінням CUDA GPU:", torch.cuda.get_device_capability()[0] >= 8)


### 12. Кількість параметрів і fp16-пам’ять

За конфігурацією Qwen3-4B-Instruct-2507 модель має **4 022 468 096 параметрів** з урахуванням спільних embedding і вихідних ваг. Теоретичні ваги fp16 займають **8 044 936 192 байти**, тобто **8,045 GB ≈ 7,492 GiB**. GB та GiB мають різні дільники. Відмінність від таблиці може виникнути через буфери, допоміжні тензори й те, що `memory_allocated()` вимірює всю поточну виділену CUDA-пам’ять процесу, а не лише суму ваг.


In [2]:
import urllib.request
import json

with urllib.request.urlopen("https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507/resolve/main/config.json") as response:
    config_4b = json.load(response)
hidden = config_4b["hidden_size"]
head_dim = config_4b["head_dim"]
attention = 2 * hidden * config_4b["num_attention_heads"] * head_dim + 2 * hidden * config_4b["num_key_value_heads"] * head_dim
feedforward = 3 * hidden * config_4b["intermediate_size"]
normalizations = 2 * hidden + 2 * head_dim
parameter_count = config_4b["vocab_size"] * hidden + config_4b["num_hidden_layers"] * (attention + feedforward + normalizations) + hidden
if not config_4b["tie_word_embeddings"]:
    parameter_count += config_4b["vocab_size"] * hidden
print("Параметрів:", f"{parameter_count:,}")
print("fp16 GB:", parameter_count * 2 / 1e9, "; GiB:", parameter_count * 2 / 2**30)
assert parameter_count == 4_022_468_096


Параметрів: 4,022,468,096
fp16 GB: 8.044936192 ; GiB: 7.492430686950684


In [ ]:
print("Параметри фактично завантаженої fp16-моделі:", PARAM_COUNT_4B)
print("fp16 з таблиці, GB:", records["fp16"]["памʼять моделі, GB"])
print("Відхилення від суми ваг, GB:", records["fp16"]["памʼять моделі, GB"] - parameter_count * 2 / 1e9)


### 13. Перплексія та випадкові слова

$$\operatorname{PPL}(x)=\exp\left(-\frac{1}{T-1}\sum_{t=2}^{T}\log p(x_t\mid x_{<t})\right).$$

Це експонента середньої втрати передбачення наступного токена. Вона залежить від токенізатора, домену, тексту й довжини доступного контексту. Навіть з одним токенізатором англійська й українська по-різному розбиваються на токени, тому їхні сирі PPL не є прямою шкалою «якості мови». Для оцінки квантизації порівнюються варіанти на однаковому тексті тієї самої мови. Випадкове вибирання слів руйнує синтаксис і семантичні зв’язки, тому зазвичай підвищує перплексію. Конкретний ефект перевіряється нижче; це не гарантоване правило для будь-якого випадкового короткого рядка.


In [ ]:
random_ppl_rows = []
for language in PPL_TEXT:
    random_ppl_rows.append({"мова": language,
        "осмислений текст, fp16": records["fp16"][f"перплексія, {language}"],
        "випадкові слова, fp16": records["fp16"]["випадкові слова, перплексія"][language]})
display(pd.DataFrame(random_ppl_rows).round(3))
for language, text in random_words.items():
    print(language + ":", text)


### 14. Пік VRAM

Під час генерації, крім ваг, потрібні KV-кеш для попередніх токенів, активації й тимчасові результати матричних операцій, logits, тензори входу/виходу та буфери квантизації/розпаковки. Додатково існує контекст CUDA й кеш алокатора; не все це потрапляє у `memory_allocated()`. `max_memory_allocated()` фіксує найбільше значення від скидання лічильника, включаючи завантаження, квантизацію, обчислення PPL та генерацію, тому це не лише «чиста генерація». `memory_reserved()` і показник `nvidia-smi` також можуть бути більшими за `memory_allocated()`.

### 15. П’ять сліпих порівнянь

Лічильник виборів наведений у відповіді 4 і секції 2.5. П’ять запитів дають дуже нестабільний висновок і не доводять еквівалентність. Навіть п’ять перемог із п’яти без нічиїх мають двостороннє p-значення точного sign-тесту $2(1/2)^5=0{,}0625$. Для орієнтовної 95% похибки частки ±10 відсоткових пунктів у найгіршому випадку потрібно близько 97 незалежних оцінок, для ±5 — 385. Це оцінка точності частки, а не універсальна гарантія виявлення будь-якої малої різниці. Потрібні різноманітні запити, попередньо визначені критерії й урахування залежності оцінок та нічиїх.

### 16. Помилки int2

Сильне округлення ваг може спричинити повторення фраз, порушення зв’язності, змішування мов, неповний переклад, неправильні числа, ігнорування формату або некоректний код. У int4 іноді залишаються слабші прояви тих самих помилок, але це потрібно перевіряти на конкретних відповідях. Нижче зібрано результати int4/int2 з тієї самої секції 2.6; після перегляду записуються фактично помічені помилки, а не очікувані симптоми.


In [ ]:
for i, prompt in enumerate(GEN_PROMPTS):
    print("ЗАПИТ", i + 1, prompt)
    for variant in ["int4", "int2"]:
        print(variant + ":", records[variant]["генерації"][i])
    print()
INT2_OBSERVATION = input("Опишіть конкретні помилки int2 та їх наявність/відсутність в int4: ").strip()
print(INT2_OBSERVATION)


### 17. Повторення на Qwen3-1.7B

Для коректного порівняння використовуються ті самі тексти, переклади, ліміти та greedy-декодування. У меншої Qwen3 режим міркування вимкнено в chat template. Порівнюються відносне зростання PPL і зміна chrF щодо власного fp16 кожної моделі. Абсолютна різниця між моделями включає також відмінності навчання: 4B-Instruct-2507 і 1.7B не є контрольованою парою лише за кількістю параметрів. Менша надлишковість може підвищити чутливість до квантизації, проте це висновок із вимірювань, а не наперед заданий результат.


In [ ]:
records_4b = records
summary_4b = summary.copy()
tokenizer_4b = tokenizer
model_id_4b = MODEL_ID
MODEL_ID = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
records_17b = {}
for name, quant_config in VARIANTS.items():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE,
        device_map=DEVICE, quantization_config=quant_config)
    model.eval()
    r = {"памʼять моделі, GB": torch.cuda.memory_allocated() / 1e9}
    for language, text in PPL_TEXT.items():
        r[f"перплексія, {language}"] = perplexity(model, text)
    r["переклади"] = [generate(model, TRANSLATE_PROMPT.format(source), 80)[0] for source, _ in TRANSLATION]
    generated = [generate(model, prompt, 200) for prompt in GEN_PROMPTS]
    r["генерації"] = [text for text, _, _ in generated]
    r["tokens/s"] = sum(n for _, n, _ in generated) / sum(seconds for _, _, seconds in generated)
    r["пік VRAM, GB"] = torch.cuda.max_memory_allocated() / 1e9
    r["chrF перекладу"] = chrf.compute(predictions=r["переклади"], references=references)["score"]
    r["BLEU перекладу"] = bleu.compute(predictions=r["переклади"], references=references)["score"]
    records_17b[name] = r
    Path("quantization_17b.json").write_text(json.dumps(records_17b, ensure_ascii=False, indent=2))
    print(name, "готово")
    del model
gc.collect()
torch.cuda.empty_cache()
tokenizer = tokenizer_4b
MODEL_ID = model_id_4b
comparison_rows = []
for size, data in [("4B", records_4b), ("1.7B", records_17b)]:
    fp_chrf = chrf.compute(predictions=data["fp16"]["переклади"], references=references)["score"]
    for variant, result in data.items():
        row = {"модель": size, "варіант": variant}
        for language in PPL_TEXT:
            metric = f"перплексія, {language}"
            row[metric] = result[metric]
            row[f"PPL / fp16, {language}"] = result[metric] / data["fp16"][metric]
        row["chrF"] = chrf.compute(predictions=result["переклади"], references=references)["score"]
        row["зміна chrF від fp16"] = row["chrF"] - fp_chrf
        comparison_rows.append(row)
display(pd.DataFrame(comparison_rows).round(3))


### 18. `OLLAMA_NUM_PARALLEL`

Параметр задає максимальну кількість одночасних запитів до однієї завантаженої моделі. Це не кількість CPU-потоків і не обов’язково кількість копій ваг. Ваги можуть бути спільними, але кожній послідовності потрібен власний KV-кеш: його бюджет приблизно пропорційний `NUM_PARALLEL × context_length`. Більша паралельність може підвищити сумарний throughput через batching, але погіршити tokens/s окремого користувача, TTFT та використання пам’яті. Запити понад доступні слоти очікують у черзі. Параметр задають перед запуском сервера й зміну перевіряють повторним вимірюванням.

### 19. Чому поширений bf16

bf16 використовує 16 біт, але має 8 біт експоненти, як fp32, і коротшу мантису. Він зберігає широкий діапазон значень та знижує ризик переповнення/зникнення малих чисел при навчанні порівняно з fp16 з 5 бітами експоненти. Сучасні прискорювачі підтримують швидкі bf16 обчислення, тому цей формат зручний для навчальних checkpoint і поширення базових ваг. Він не точніший за fp16 за мантисою й не є єдиним форматом: для запуску широко використовуються fp16, fp8, int8 та 4-бітні ваги; конкретний тип потрібно перевіряти в конфігурації моделі.

### 20. OpenAI-сумісний API й ChatGPT

Сумісні HTTP-маршрути, структура `messages`, JSON-відповіді та streaming дозволяють використовувати той самий клієнтський код із різними серверами, змінивши `base_url` і назву моделі. Це зменшує кількість окремих інтеграцій. Сумісність зазвичай часткова, тому підтримку параметрів, tools, JSON Schema та полів reasoning потрібно перевіряти для конкретного рантайму. OpenAI API став поширеним інтерфейсом сервісів мовних моделей; ChatGPT є окремим застосунком OpenAI. Запит до `localhost` із таким API виконує локальна модель, без звернення до ChatGPT чи хмарного API OpenAI.


## 1.8. Мій звіт

**Виконав: Некрасов Артур Дмитрович.**

**Залізо.** Цей запуск має позначку `user_computer`. Для початкового `execution_environment` це доступне середовище виконання, а не особистий ПК студента. Його результати не підтверджують характеристики іншого комп’ютера.

| Характеристика | Значення | Джерело |
| --- | --- | --- |
| Середовище | user_computer | manifest_after.json |
| CPU | AMD Ryzen 7 7735HS with Radeon Graphics | Get-CimInstance Win32_Processor |
| Логічні CPU | 16 | os.cpu_count(); у рантаймі використано 4 потоки |
| RAM | 16.328 GB / 15.207 GiB | psutil.virtual_memory() |
| GPU/VRAM | NVIDIA GeForce RTX 4060 Laptop GPU, 8188 | nvidia-smi --query-gpu=name,memory.total |
| Ефективне читання RAM | 11.152 GB/s | Медіана 9 послідовних numpy.sum масиву 192 MB float64; приблизна ефективна пропускна здатність RAM, не паспортна характеристика |
| Паспортна пропускна здатність | Не визначена | Потрібна специфікація особистого CPU/RAM або GPU |

**Розрахунок.** Qwen3-1.7B має 1 720 574 976 параметрів. Ідеальний бюджет ваг: $M_{weights}=N\cdot b/8$. Для fp16 KV-кешу: $M_{KV}=2\cdot L\cdot B\cdot T\cdot H_{KV}\cdot d_{head}\cdot 2$. Тут $L=28$, $T=4096$, $H_{KV}=8$, $d_{head}=128$; $B$ — кількість одночасних послідовностей. На одну послідовність потрібно 469,762,048 байтів, тобто 0.4375 GiB. Ваги спільні для клієнтів, KV-кеш — окремий.

| Варіант | Ідеальні ваги, GiB | KV, 1 клієнт / 2 клієнти, GiB | Разом із запасом 1 GiB, 2 клієнти |
| --- | --- | --- | --- |
| fp16 | 3.205 | 0.438 / 0.875 | 5.080 GiB |
| int8 | 1.602 | 0.438 / 0.875 | 3.477 GiB |
| int4 | 0.801 | 0.438 / 0.875 | 2.676 GiB |

Для порівняння, 4B у fp16: 7,492 GiB ваг + 1,125 GiB KV-кешу двох клієнтів + 1 GiB рантайму + 2 GiB ОС ≈ **11,62 GiB**, що перевищує доступні 9,73 GiB початкового середовища. Для 4B int4 ідеальний сумарний бюджет ≈ **6,00 GiB**, але потрібні поправка на метадані й фактичні буфери, а читати ваги доведеться довше. Тому для інтерактивного застосунку перевірено меншу 1.7B. Для іншого ПК цей висновок потрібно перерахувати.

Окремо залишено 2 GiB для ОС; 1 GiB запасу рантайму є оцінкою, яку перевіряємо фактичним RSS. Справжній GGUF містить масштаби, метадані та змішані типи ваг, тому його розмір більший за ідеальні 4 біти на параметр. Для моделей, які не поміщаються повністю у VRAM, потрібно окремо враховувати RAM та передачу між CPU і GPU.

У моделі пропускної здатності для одного користувача орієнтир швидкості — $v\approx bandwidth/M_{weights}$. У таблиці використано фактичний розмір GGUF та виміряне читання RAM:

| Варіант | GGUF, GB | Пік RSS процесу, GB | Орієнтир bandwidth / GGUF, tok/s |
| --- | --- | --- | --- |
| Q8_0 | 1.834 | 2.906 | 6.1 |
| Q4_K_M | 1.107 | 2.856 | 10.1 |

Це умовна оцінка, а не доведена апаратна стеля: `numpy.sum` не вимірює максимальну паспортну пропускну здатність контролера пам’яті. Розмір файлу також не дорівнює точному обсягу читання ваг на кожному токені. Через різні ядра, кеші й характер доступу виміряна генерація може бути вищою за такий орієнтир. Для остаточного розрахунку особистого ПК потрібні модель CPU/GPU, конфігурація пам’яті та специфікація її пропускної здатності. Обчислення, розпаковка, KV-кеш і обробка промпту зазвичай віддаляють результат від ідеальної bandwidth-межі.

**Обрана модель, квантизація й рантайм.** Qwen3-1.7B у GGUF; початково Q8_0, після оптимізації Q4_K_M. Це постнавчена модель із chat template; режим міркування вимкнено, бо для коротких карток важливі затримка й передбачуваний JSON. Рантайм — llama.cpp b12110 (`0.9.0-dev`, commit `cd959bae9`), CPU, 4 потоки, 2 слоти по 4096 токенів. Однакові ваги за походженням і однакові налаштування дозволяють перевіряти вплив квантизації. Файли взято з `unsloth/Qwen3-1.7B-GGUF`, revision `d7f544eead698dbd1f15126ef60b45a1e1933222`; SHA-256 наведено в `app/data/manifest_before.json` та `manifest_after.json`.

**Вимірювання до оптимізації та після.** Використано лекційний `bench.py` без змін. Перед вимірюванням є прогрів; по 8 запитів для 1, 2, 4 клієнтів; ліміт 128 нових токенів, temperature=0, reasoning_effort=none. Prompt cache вимкнено в обох запусках. TTFT включає очікування черги, обробку промпту та перший видимий токен. Модель має 2 слоти, тому при 4 клієнтах зростає черга.

| Мітка | Клієнти | Запити | Помилки | TTFT p50 / p95, с | tok/s на користувача p50 | Сумарний tok/s |
| --- | --- | --- | --- | --- | --- | --- |
| before | 1 | 8 | 0 | 0.461 / 0.759 | 20.0 | 17.9 |
| before | 2 | 8 | 0 | 0.679 / 0.869 | 14.1 | 23.7 |
| before | 4 | 8 | 0 | 7.508 / 11.592 | 13.5 | 22.7 |
| after | 1 | 8 | 0 | 0.282 / 0.401 | 29.5 | 27.3 |
| after | 2 | 8 | 0 | 0.391 / 0.529 | 19.1 | 34.3 |
| after | 4 | 8 | 0 | 5.423 / 8.175 | 19.0 | 32.8 |

Для одного клієнта: 20.0 → 29.5 tok/s, прискорення **1.48×**. Розмір GGUF зменшився у **1.66×**. Пік RSS у таблиці не є чистим розміром ваг: він включає відображені сторінки, KV-кеш, буфери, можливе перепакування та алокації рантайму. Це короткий одинарний запуск, тому p95 з 8 спостережень і різниця на частки секунди мають обмежену статистичну надійність.

**Застосунок.** «Картки з конспекту» перетворює власний текст на 1–5 запитань із відповідями. Є CLI (`cli.py`) та вебінтерфейс (`server.py`, `web/`). Обидва використовують один модуль `core.py` і локальний OpenAI-сумісний HTTP API. Послідовність: перевірка довжини → системний промпт і few-shot приклад → JSON Schema → перевірка кількості й полів → перевірка дослівної наявності відповіді у джерелі → показ/експорт. Дані конспекту не є інструкціями. Перевірка цитат не доводить відповідності запитання відповіді, тому потрібна змістова перевірка.

Параметри: **temperature=0, seed=1**, щоб зменшити варіативність і перефразування; **max_tokens=640**, щоб обмежити час і запобігти нескінченному продовженню; **контекст 4096 на слот**, достатній для короткого конспекту, службового промпту й відповіді. Вхід обмежено 1800 символами та 2800 байтами UTF-8; при довгому тексті його потрібно ділити на частини. Режим міркування вимкнено. Таймаут запиту 180 с; автоматичних повторних генерацій немає. CLI показує лічильник очікування, вебінтерфейс — індикатор і час. Оброблено відсутній сервер, таймаут, некоректний JSON, досягнення ліміту токенів та відповідь без дослівного підтвердження. JSON Schema перевіряє форму, а не фактичну правильність.

**Тестові сценарії й результати.** Ті самі 10 конспектів та очікування збережено у `app/data/prompts.json`, повні відповіді — у `quality_before.json` та `quality_after.json`. Позначка «Пройдено» в таблиці означає лише формальні перевірки й дослівну цитату.

| № | Сценарій | Правильний результат | before | after |
| --- | --- | --- | --- | --- |
| 1 | Квантизація | Дві різні картки; правильні біти/байти; відповіді дослівно з тексту. | Пройдено | Пройдено |
| 2 | Мережі | Не плутати 404 і 503; дві різні картки з цитатами. | Пройдено | Пройдено |
| 3 | Заперечення | Не втрачати заперечення та не приписувати sort повернення нового списку. | Пройдено | Пройдено |
| 4 | Числа й одиниці | Зберегти числа, одиниці та відмінність контрольного зразка. | Пройдено | Відхилено валідатором |
| 5 | Алгоритми | Не змішувати передумови й складності двох алгоритмів. | Відхилено валідатором | Відхилено валідатором |
| 6 | Недостатній контекст | Одна картка про наявний факт; не вигадувати тему чи дедлайн роботи. | Пройдено | Пройдено |
| 7 | Суперечність | Уточнити версію у запитанні або зберегти невизначеність; не називати остаточну дату. | Пройдено | Пройдено |
| 8 | Інструкція всередині даних | Повернути картки у JSON; не виконувати інструкцію з конспекту. | Пройдено | Пройдено |
| 9 | Змішані мови | Українські запитання; терміни й цитати зберегти; не плутати validation та test. | Пройдено | Пройдено |
| 10 | Незвична назва й факт | Не замінювати вигадану назву знайомим пристроєм; правильно зіставити кольори й вимірювання. | Пройдено | Пройдено |

Формальні перевірки: **before 9/10, after 8/10**. Ця цифра не є оцінкою всього змісту. Зокрема, запитання «Як DNS перетворює…» потребує пояснення механізму, якого немає у відповіді-цитаті; картка про «теоретичні висновки/аналіз» також додає деталь, якої джерело прямо не встановлює. Такі картки можуть пройти перевірку цитат і залишитися невдалими для навчання. Змістові спостереження наведено в `app/data/quality_review.json`. Результати малого набору не доводять загальну рівність якості квантизацій.

**Помилки моделі та виправлення.** У попередньому запуску Q8_0 модель заповнила `question` твердженнями без знака питання. Додано явну вимогу запитань, приклад правильної пари та перевірку завершення на `?`; цей промпт однаковий у всіх основних before/after перевірках. Первинну відповідь збережено у `failure_before_prompt_fix.json`. На конспекті про алгоритми обидві моделі перефразували відповідь: «Часова складність бінарного пошуку становить O(log n).» замість наявного «Його часова складність становить O(log n).». Зміст близький, але вимога дослівної цитати порушена; застосунок чесно відхиляє результат. Запитання зі знаком `?` теж не завжди є змістовно добрим: повністю виправити цю проблему перевіркою форми не вдалося.

**Перевірка реалізації й демонстрація.** `app/tests.py` перевіряє валідну картку, порожній/надмірний вхід, некоректний JSON, непідтверджену відповідь, твердження замість запитання, обробку таймауту SDK, реальну помилку підключення та HTTP-відповіді. Тест таймауту є модульним із змодельованим винятком SDK; інтеграційний тест успіху використовує справжню локальну модель. Результат — `app/data/test_results.json`. В `demo/app_demo.mp4` відтворено фактичний stdout CLI з часовими мітками; це не захоплення робочого столу. Показано успіх, порожній вхід, відсутній рантайм і відхилення недослівної відповіді. Першоджерело відео — `app_demo.cast`, кадри — `*_cli.png`. Вебінтерфейс потрібно додатково перевірити у браузері на своєму комп’ютері.

**Висновок.** Для цього запуску обрано Q4_K_M: він менший і швидший, а частка проходження перевірок цитат на 10 конспектах не зменшилася. Залишилися змістові й мовні помилки, які автоматична перевірка структури не виявляє. Для остаточної здачі потрібно виконати GPU-демо в Colab T4, ввести власні сліпі оцінки та провести вимірювання на особистому комп’ютері; дані поточного середовища не замінюють ці пункти.


## Джерела

- [Qwen3-4B-Instruct-2507: конфігурація й картка моделі](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507)
- [Qwen3-1.7B: модель і режим міркування](https://huggingface.co/Qwen/Qwen3-1.7B)
- [Hugging Face: перплексія](https://huggingface.co/docs/transformers/en/perplexity)
- [Hugging Face: Tokenizers](https://huggingface.co/learn/llm-course/en/chapter6/1)
- [Hugging Face: Quanto](https://huggingface.co/docs/transformers/main/quantization/quanto)
- [NVIDIA: compute capability GPU](https://developer.nvidia.com/cuda/gpus)
- [NVIDIA: bf16 і compute capability](https://nvidia.github.io/numba-cuda/reference/types.html)
- [Ollama FAQ: контекст і паралельність](https://docs.ollama.com/faq)
- [Ollama: OpenAI-сумісність](https://docs.ollama.com/api/openai-compatibility)
- [llama.cpp: сервер і JSON Schema](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md)
